triage: subject_id,stay_id, (ignore rest)

pyxis: subject_id,stay_id,charttime,med_rn,name,gsn_rn,gsn

medrecon: subject_id,stay_id,charttime,name,gsn,ndc,etc_rn,etccode,etcdescription

edstays: subject_id,hadm_id,stay_id,intime,outtime,gender,race,arrival_transport,disposition


Each of the above file names are for example referncable: "mimic-iv-ed-2.2/ed/medrecon.csv" and similarly for the others

In [1]:
import pandas as pd
import numpy as np
import requests
import time
from sklearn.preprocessing import MinMaxScaler

# Cleaning data
triage_df = pd.read_csv('mimic-iv-ed-2.2/ed/triage.csv')
pyxis_df = pd.read_csv('mimic-iv-ed-2.2/ed/pyxis.csv')
medrecon_df = pd.read_csv('mimic-iv-ed-2.2/ed/medrecon.csv')
edstays_df = pd.read_csv('mimic-iv-ed-2.2/ed/edstays.csv')

def norm(x):
    return str(x).lower().strip() if pd.notna(x) else None

# Normalize meds
medrecon_df['medicine'] = medrecon_df['name'].apply(norm)
medrecon_df['gsn'] = medrecon_df['gsn'].apply(norm)

pyxis_df['medicine'] = pyxis_df['name'].apply(norm)
pyxis_df['gsn'] = pyxis_df['gsn_rn'].apply(norm)

# Normalize NDC (helper function to clean NDC codes)
def normalize_ndc(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s

medrecon_df['ndc_clean'] = medrecon_df['ndc'].apply(normalize_ndc)

FileNotFoundError: [Errno 2] No such file or directory: 'mimic-iv-ed-2.2/ed/triage.csv'

Normalising the duration

In [ ]:
# duration calculations
edstays_df['intime'] = pd.to_datetime(edstays_df['intime'], errors='coerce')
edstays_df['outtime'] = pd.to_datetime(edstays_df['outtime'], errors='coerce')

edstays_df['duration_hours'] = (
    (edstays_df['outtime'] - edstays_df['intime'])
    .dt.total_seconds() / 3600
)

edstays_df = edstays_df[
    edstays_df['duration_hours'].notna() &
    (edstays_df['duration_hours'] > 0)
]

Matching the medication sources across, GSN Frequency mapping & grade of medication

In [ ]:
# combining sources
meds_df = pd.concat([
    medrecon_df[['subject_id', 'stay_id', 'medicine', 'gsn', 'ndc_clean']],
    pyxis_df[['subject_id', 'stay_id', 'medicine', 'gsn']].assign(ndc_clean=None)
], ignore_index=True)

meds_df = meds_df[meds_df['medicine'].notna()]

df = meds_df.merge(
    edstays_df[['subject_id', 'stay_id', 'duration_hours']],
    on=['subject_id', 'stay_id'],
    how='inner'
)

# freq mapping
df['gsn_fill'] = df['gsn'].fillna('<<missing>>')

combo_freq = (
    df.groupby(['medicine', 'gsn_fill'])
    .size()
    .reset_index(name='combo_freq')
)

df = df.merge(combo_freq, on=['medicine', 'gsn_fill'], how='left')

# NDC normalization for OpenFDA lookup
def normalize_ndc_openfda(ndc):
    if pd.isna(ndc):
        return None
    s = str(ndc).strip().replace('-', '')

    if s.endswith('.0'):
        s = s[:-2]

    # pad to 11 digits (OpenFDA standard)
    if len(s) == 10:
        s = '0' + s

    if len(s) != 11:
        return None

    return s

df['ndc_clean'] = df['ndc_clean'].apply(normalize_ndc_openfda)
# openfda lookup
def fetch_openfda(ndc):
    url = f"https://api.fda.gov/drug/ndc.json?api_key=APIKeyHere&search=product_ndc:\"{ndc}\"&limit=1" # Add your own openFDA API key here
    try:
        r = requests.get(url, timeout=5)
        r.raise_for_status()
        data = r.json()
        results = data.get('results', [])
        if not results:
            return None

        openfda = results[0].get('openfda', {})

        return {
            'ndc': ndc,
            'rxcui': openfda.get('rxcui'),
            'generic_name': openfda.get('generic_name'),
            'brand_name': openfda.get('brand_name'),
        }
    except:
        return None

unique_ndcs = df['ndc_clean'].dropna().unique()[:300]

rows = []
for ndc in unique_ndcs:
    res = fetch_openfda(ndc)
    if res:
        rows.append(res)
    time.sleep(0.1)

# safe openfda DataFrame construction
expected_cols = ['ndc', 'rxcui', 'generic_name', 'brand_name']

openfda_df = pd.DataFrame(rows)

for col in expected_cols:
    if col not in openfda_df.columns:
        openfda_df[col] = pd.NA

print("NDCs queried:", len(unique_ndcs))
print("OpenFDA matches:", openfda_df['ndc'].notna().sum())

# safe merge (no crash)
if not openfda_df.empty:
    df = df.merge(
        openfda_df,
        left_on='ndc_clean',
        right_on='ndc',
        how='left'
    )
else:
    print("No OpenFDA data; applying fallback only")
    df['rxcui'] = pd.NA
    df['generic_name'] = pd.NA
    df['brand_name'] = pd.NA

# med grading
df['has_gsn'] = df['gsn'].notna().astype(float)
df['has_ndc'] = df['ndc_clean'].notna().astype(float)
df['has_rxcui'] = df['rxcui'].notna().astype(float)

# Fallback signal (if OpenFDA fails)
df['fallback_signal'] = (
    df['has_gsn'] * 0.7 +
    df['has_ndc'] * 0.3
)

# Use RxCUI if available, otherwise fallback
df['identifier_strength'] = np.where(
    df['has_rxcui'] == 1,
    1.0,  
    df['fallback_signal']    
)

# OpenFDA richness
df['has_generic'] = df['generic_name'].notna().astype(float)
df['has_brand'] = df['brand_name'].notna().astype(float)

df['openfda_quality'] = (
    df['has_generic'] * 0.6 +
    df['has_brand'] * 0.4
)

# freq norm
freq = df['combo_freq'].fillna(0)

df['freq_norm'] = (
    (freq - freq.min()) / (freq.max() - freq.min())
    if freq.max() > freq.min() else 0
)

df['specificity'] = 1 - df['freq_norm']

# final grade
df['grade_of_medicine'] = (
    df['identifier_strength'] * 0.5 +
    df['openfda_quality'] * 0.3 +
    df['specificity'] * 0.2
)

df['grade_norm'] = MinMaxScaler().fit_transform(df[['grade_of_medicine']])

Final duration normalisation

In [ ]:
# duration normalization
dur = df['duration_hours'].clip(lower=0)
p99 = dur.quantile(0.99)
dur = dur.clip(upper=p99)

df['duration_norm'] = MinMaxScaler().fit_transform(dur.to_frame())

LOC Calculation & output

In [ ]:
# LOC score calculation
df['loc_score'] = (
    df['duration_norm'] * 0.4 +
    df['grade_norm'] * 0.6
)
df['loc_score_0_100'] = (df['loc_score'] * 100).clip(0, 100)

# output
final_df = df[[
    'subject_id',
    'stay_id',
    'medicine',
    'gsn',
    'ndc_clean',
    'rxcui',
    'duration_hours',
    'duration_norm',
    'combo_freq',
    'grade_norm',
    'loc_score_0_100'
]]
final_df.to_csv('ed_loc_final.csv', index=False)

print("Final shape:", final_df.shape)
print(final_df.head())